In [3]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("/data/processed/ecommerce_transactions.csv")
df.head()


,order_id,user_id,age,gender,city,product_id,category,price_x,quantity,total_amount,timestamp
0,1,609,57,Male,Chennai,21,Fashion,2597,2,5194,2024-09-26
1,2,84,35,Male,Delhi,215,Sports,16899,3,50697,2024-11-04
2,3,598,35,Male,Mumbai,290,Electronics,21625,3,64875,2024-06-19
3,4,324,47,Female,Mumbai,134,Electronics,21753,2,43506,2023-07-29
4,5,686,55,Female,Mumbai,161,Home,5343,3,16029,2023-09-02


In [4]:
user_item_matrix = df.pivot_table(
    index="user_id",
    columns="product_id",
    values="quantity",
    aggfunc="sum",
    fill_value=0
)
user_item_matrix.shape


(1000, 300)

In [5]:
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)


In [6]:
def recommend_products(user_id, n_recommendations=5):
    # Similar users (excluding self)
    similar_users = (
        user_similarity_df[user_id]
        .sort_values(ascending=False)
        .iloc[1:11]
    )

    # Products bought by similar users
    similar_users_purchases = user_item_matrix.loc[similar_users.index]

    # Weighted sum of interactions
    weighted_scores = np.dot(
        similar_users.values,
        similar_users_purchases
    )

    scores = pd.Series(
        weighted_scores,
        index=user_item_matrix.columns
    )

    # Remove products already purchased
    purchased_products = user_item_matrix.loc[user_id]
    scores = scores[purchased_products == 0]

    return scores.sort_values(ascending=False).head(n_recommendations)


In [7]:
recommend_products(user_id=10)


,0
product_id,
84,2.015994
265,1.745017
26,1.677659
242,1.501029
178,1.145377


In [8]:
products = df[["product_id", "category", "price_x"]].drop_duplicates()

def recommend_with_details(user_id, n=5):
    recs = recommend_products(user_id, n)
    return products[products["product_id"].isin(recs.index)]
recommend_with_details(10)

,product_id,category,price_x
39,84,Electronics,18421
170,265,Beauty,5263
327,26,Sports,4600
650,178,Sports,2907
726,242,Fashion,1398


In [10]:
segments = pd.read_csv('/data/processed/mall_customers_segmented.csv')
segments = segments[['CustomerID', 'Spending Score (1-100)']]
segments.columns = ['user_id', 'segment']
df = df.merge(segments, on='user_id', how='left')

In [11]:
segment_popularity = (
    df.groupby(["segment", "product_id"])
    .agg(total_quantity=("quantity", "sum"))
    .reset_index()
)
def recommend_by_segment(user_id, n=5):
    user_segment = df[df["user_id"] == user_id]["segment"].iloc[0]

    return (
        segment_popularity[segment_popularity["segment"] == user_segment]
        .sort_values("total_quantity", ascending=False)
        .head(n)
        .merge(products, on="product_id")
    )
recommend_by_segment(10)


,segment,product_id,total_quantity,category,price_x
0,47,232,10,Beauty,389
1,47,112,9,Sports,6374
2,47,172,8,Fashion,3500
3,47,243,8,Home,7496
4,47,61,7,Fashion,6770


In [12]:
def hybrid_recommendation(user_id, n=5):
    try:
        return recommend_with_details(user_id, n)
    except:
        return recommend_by_segment(user_id, n)


In [13]:
import os

os.makedirs('../data/processed', exist_ok=True)
user_item_matrix.to_csv("../data/processed/user_item_matrix.csv")

### Recommendation Strategy

1. Used cosine similarity on user-item interactions.
2. Identified similar users based on purchasing behavior.
3. Recommended products not yet purchased by the user.
4. Enhanced recommendations using customer segments.
5. Implemented fallback logic for cold-start users.
